# `06 — Binary Heap (Min-Heap)`

We study:
- **Heap invariant**
- **SiftUp / SiftDown** (restore invariant)
- How to store heap in a **vector** (Python list)
- **push / pop / remove(i)**
- **build heap (heapify)** and why it is **O(N)**

Goal:
- Be able to implement and trace operations
- Know the **Big-O** of each operation (oral exam classic)


## `1. What problem does heap solve? `

### `1.1 Motivation`

We want a structure supporting:
- **push(x)**
- **remove(i)** (remove element if we know its index)
- **find_min()**
- **len()**

A heap is a compromise:
- **find_min() is O(1)** (root)
- updates are **O(log N)** (tree height)

> (See the comparison table in slides.)

### `1.2 Complexity table`

| Operation | Heap |
|---|---:|
| **find_min()** | **O(1)** |
| **push(x)** | **O(log N)**|
| **pop_min()** | **O(log N)**|
| **remove(i)** | **O(log N)**|
| **heapify(data)** | **O(N)**|


## `2. Heap invariant`

### `2.1 Heap invariant (min-heap)`

We store elements in a binary tree.

**Invariant:**
For every node:
**parent ≤ child**

So the root is the **minimum element**.

Example:
```bash
        x0
      /    \
    x1      x2
   / \     / \
 x3  x4  x5  x6

Constraints:
x0 ≤ x1, x0 ≤ x2
x1 ≤ x3, x1 ≤ x4
x2 ≤ x5, x2 ≤ x6
...
```




## `3. Store heap in a vector (Python list)`

### `3.1 Store heap in an array`

We enumerate nodes level by level:

index:  0  1  2  3  4  5  6 ...
value: [x0 x1 x2 x3 x4 x5 x6 ...]

Formulas:
- **parent(i) = (i - 1) // 2**
- **left(i) = 2*i + 1**
- **right(i) = 2*i + 2**



## `4. Restore invariant: SiftUp and SiftDown`


### `4.1 When invariant breaks`

- If a value becomes **smaller** than its parent → fix by moving it **up** (**SiftUp**)
- If a value becomes **larger** than one of its children → fix by moving it **down** (**SiftDown**)


In [5]:
from typing import Self, List
from dataclasses import dataclass, field


def parent(i: int) -> int:
    return (i - 1) // 2

def left(i: int) -> int:
    return 2 * i + 1

def right(i: int) -> int:
    return 2 * i + 2


@dataclass
class MinHeap:

    data: List[int] = field(default_factory=list)

    # -----------------------
    #      Visualization:
    # -----------------------

    def show(self: Self, label: str = "") -> None:
        prefix = f"{label} " if label else ""
        print(f"{prefix}Heap(size={len(self.data)}): {self.data}")

    def show_tree(self: Self, label: str = "") -> None:
        """
        Print heap levels (rough tree view).
        """
        prefix = f"{label}\n" if label else ""
        print(prefix, end="")

        n: int = len(self.data)
        level: int = 0
        i: int = 0
        while i < n:
            count: int = 2 ** level
            row = self.data[i:i + count]
            print(f"level {level}: {row}")
            i += count
            level += 1

    # -----------------------
    #     Core Operations:
    # -----------------------

    def find_min(self: Self) -> int:
        if not self.data:
            raise IndexError("find_min from empty heap")
        return self.data[0]

    def sift_up(self: Self, i: int, *, verbose: bool = False) -> None:
        # Slide idea: while not root and data[i] < data[parent(i)] swap and go up.
        while i > 0:
            p: int = parent(i)
            if self.data[i] < self.data[p]:
                if verbose:
                    print(f"  sift_up: swap i={i} val={self.data[i]} with p={p} val={self.data[p]}")
                self.data[i], self.data[p] = self.data[p], self.data[i]
                i = p
            else:
                break

    def sift_down(self: Self, i: int, *, verbose: bool = False) -> None:
        # Slide idea: choose smaller child and swap if child < parent.
        n: int = len(self.data)
        while True:
            l: int = left(i)
            r: int = right(i)

            if l >= n:
                return  # no children

            # choose min child
            cmin: int = l
            if r < n and self.data[r] < self.data[l]:
                cmin = r

            if self.data[cmin] < self.data[i]:
                if verbose:
                    print(f"  sift_down: swap i={i} val={self.data[i]} with c={cmin} val={self.data[cmin]}")
                self.data[i], self.data[cmin] = self.data[cmin], self.data[i]
                i = cmin
            else:
                return

    def push(self: Self, x: int, *, verbose: bool = False) -> None:
        # data.append(x); sift_up(last).
        self.data.append(x)
        if verbose:
            print(f"push({x}) -> appended at i={len(self.data)-1}")
        self.sift_up(len(self.data) - 1, verbose=verbose)

    def pop_min(self: Self, *, verbose: bool = False) -> int:
        # swap root with last; pop last; sift_down(root).
        if not self.data:
            raise IndexError("pop_min from empty heap")

        if len(self.data) == 1:
            return self.data.pop()

        if verbose:
            print(f"pop_min(): swap root {self.data[0]} with last {self.data[-1]}")

        self.data[0], self.data[-1] = self.data[-1], self.data[0]
        res: int = self.data.pop()

        self.sift_down(0, verbose=verbose)
        return res

    def remove(self: Self, i: int, *, verbose: bool = False) -> int:
        """
        Remove element at index i.
        Slide version: swap with last, pop, then sift_up and sift_down from i.
        """
        n: int = len(self.data)
        if i < 0 or i >= n:
            raise IndexError("remove index out of range")

        if n == 1:
            return self.data.pop()

        if verbose:
            print(f"remove(i={i}): swap data[i]={self.data[i]} with last={self.data[-1]}")

        self.data[i], self.data[-1] = self.data[-1], self.data[i]
        removed: int = self.data.pop()

        # i might now contain a different value; fix both directions
        if i < len(self.data):
            self.sift_up(i, verbose=verbose)
            self.sift_down(i, verbose=verbose)

        return removed

    def heapify(self: Self, arr: List[int], *, verbose: bool = False) -> None:
        """
        Build heap in O(N) using backward sift_down from last parent.
        """
        self.data = arr.copy()
        n: int = len(self.data)

        # last node with a child: parent(n-1) = (n-2)//2
        start: int = (n - 2) // 2
        if verbose:
            print(f"heapify: start from i={start} down to 0")

        for i in range(start, -1, -1):
            if verbose:
                print(f" sift_down from i={i} val={self.data[i]}")
            self.sift_down(i, verbose=verbose)



In [6]:
# Building the heap with push:

print("-" * 55)
print(f"{'Building heap using push() (watch growth)':^55}")
print("-" * 55)

h = MinHeap()
h.show(label="Start")

for x in [5, 3, 10, 6, 7, 12, 11, 9]:
    h.push(x, verbose=True)
    h.show(label=f"after push({x})")
    h.show_tree(label="tree view")
    print("-" * 55)

print("find_min:", h.find_min())

-------------------------------------------------------
       Building heap using push() (watch growth)       
-------------------------------------------------------
Start Heap(size=0): []
push(5) -> appended at i=0
after push(5) Heap(size=1): [5]
tree view
level 0: [5]
-------------------------------------------------------
push(3) -> appended at i=1
  sift_up: swap i=1 val=3 with p=0 val=5
after push(3) Heap(size=2): [3, 5]
tree view
level 0: [3]
level 1: [5]
-------------------------------------------------------
push(10) -> appended at i=2
after push(10) Heap(size=3): [3, 5, 10]
tree view
level 0: [3]
level 1: [5, 10]
-------------------------------------------------------
push(6) -> appended at i=3
after push(6) Heap(size=4): [3, 5, 10, 6]
tree view
level 0: [3]
level 1: [5, 10]
level 2: [6]
-------------------------------------------------------
push(7) -> appended at i=4
after push(7) Heap(size=5): [3, 5, 10, 6, 7]
tree view
level 0: [3]
level 1: [5, 10]
level 2: [6, 7]
------

**Note takeaway**: Each push can move up at most the tree height `H=log_2 N`, so push is `O(log N)`.

In [7]:
# Pop min repeatedly:

print("-" * 55)
print(f"{'Shrinking heap using pop_min() (watch shrink)':^55}")
print("-" * 55)

h = MinHeap()
for x in [5, 3, 10, 6, 7, 12, 11, 9]:
    h.push(x)

h.show(label="Initial heap")
h.show_tree(label="Initial tree")

while len(h.data) > 0:
    m = h.pop_min(verbose=True)
    h.show(label=f"pop_min() -> {m}")
    h.show_tree(label="tree view")
    print("-" * 55)
    

-------------------------------------------------------
     Shrinking heap using pop_min() (watch shrink)     
-------------------------------------------------------
Initial heap Heap(size=8): [3, 5, 10, 6, 7, 12, 11, 9]
Initial tree
level 0: [3]
level 1: [5, 10]
level 2: [6, 7, 12, 11]
level 3: [9]
pop_min(): swap root 3 with last 9
  sift_down: swap i=0 val=9 with c=1 val=5
  sift_down: swap i=1 val=9 with c=3 val=6
pop_min() -> 3 Heap(size=7): [5, 6, 10, 9, 7, 12, 11]
tree view
level 0: [5]
level 1: [6, 10]
level 2: [9, 7, 12, 11]
-------------------------------------------------------
pop_min(): swap root 5 with last 11
  sift_down: swap i=0 val=11 with c=1 val=6
  sift_down: swap i=1 val=11 with c=4 val=7
pop_min() -> 5 Heap(size=6): [6, 7, 10, 9, 11, 12]
tree view
level 0: [6]
level 1: [7, 10]
level 2: [9, 11, 12]
-------------------------------------------------------
pop_min(): swap root 6 with last 12
  sift_down: swap i=0 val=12 with c=1 val=7
  sift_down: swap i=1 val=12 w

**Note takeaway**: pop_min does one swap + one sift_down, which is `O(log N)`.

In [ ]:
# Remove arbitrary index remove(i):

print("-" * 55)
print(f"{'Remove(i) demo (swap+pop then sift_up/sift_down)':^55}")
print("-" * 55)

h = MinHeap()
h.heapify([7, 9, 8, 10, 12, 11, 5, 3, 6], verbose=True)  # Any array
h.show(label="After heapify")
h.show_tree(label="tree view")

# Remove an internal node index:
idx_to_remove = 3
removed = h.remove(idx_to_remove, verbose=True)

h.show(label=f"remove(i={idx_to_remove}) -> removed={removed}")
h.show_tree(label="tree view")


-------------------------------------------------------
   Remove(i) demo (swap+pop then sift_up/sift_down)    
-------------------------------------------------------
heapify: start from i=3 down to 0
 sift_down from i=3 val=10
  sift_down: swap i=3 val=10 with c=7 val=3
 sift_down from i=2 val=8
  sift_down: swap i=2 val=8 with c=6 val=5
 sift_down from i=1 val=9
  sift_down: swap i=1 val=9 with c=3 val=3
  sift_down: swap i=3 val=9 with c=8 val=6
 sift_down from i=0 val=7
  sift_down: swap i=0 val=7 with c=1 val=3
  sift_down: swap i=1 val=7 with c=3 val=6
After heapify Heap(size=9): [3, 6, 5, 7, 12, 11, 8, 10, 9]
tree view
level 0: [3]
level 1: [6, 5]
level 2: [7, 12, 11, 8]
level 3: [10, 9]
remove(i=3): swap data[i]=7 with last=9
remove(i=3) -> removed=7 Heap(size=8): [3, 6, 5, 9, 12, 11, 8, 10]
tree view
level 0: [3]
level 1: [6, 5]
level 2: [9, 12, 11, 8]
level 3: [10]


## `7. Building heap (heapify) and why it is O(N)`


### `7.1 heapify(data)`

**Idea from slides:**
Call `sift_down` from the last parent node down to the root.

Last node with a child:
**parent(N-1) = (N-2)//2**

Pseudo:
for i from (N-2)//2 down to 0:
    sift_down(i)

**Important result:**
heapify is **O(N)**, not O(N log N)!


### **Why heapify is O(N)**

Even though a single `sift_down` can take $O(\log N)$,
most nodes are near the bottom and have **very small subtree height**.

- Many nodes have height 0 or 1 (cheap)
- Only a few nodes are near the top (expensive)

The slide sums this as:

$$
T \le N \sum_{h=0}^{\infty} \frac{h}{2^h} = 2N = O(N)
$$

**Intuition:**
- About $\frac{N}{2}$ nodes are leaves (height 0) → 0 work
- About $\frac{N}{4}$ nodes have height 1 → 1 step each
- About $\frac{N}{8}$ nodes have height 2 → 2 steps each
- ...

Total work: $\frac{N}{2} \cdot 0 + \frac{N}{4} \cdot 1 + \frac{N}{8} \cdot 2 + ... = O(N)$

## `8. Python built-ins`

Python has `heapq` which implements a **min-heap** on a list:

- `heapq.heapify(data)`   → build heap
- `heapq.heappush(data,x)` → push
- `heapq.heappop(data)`    → pop_min